# 🌾 AgriScore KZ — Exploratory Data Analysis

**Датасет:** Выгрузка по выданным субсидиям 2025 год (обезличенная)

**Цель:** Визуальный анализ распределения субсидий по регионам, направлениям и статусам одобрения.
Выявление ключевых паттернов для построения ML-модели скоринга заявок.

---
| Раздел | Содержание |
|--------|------------|
| 1 | Загрузка и очистка данных |
| 2 | Распределение заявок по областям |
| 3 | Распределение сумм субсидий |
| 4 | Топ-10 районов по количеству заявок |
| 5 | Соотношение одобренных / отклонённых |
| 6 | Средняя сумма по направлениям животноводства |
| 7 | Корреляционная матрица |
| 8 | Выводы |

## 0. Настройка окружения и загрузка данных

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import FancyBboxPatch
import matplotlib.patches as mpatches

# ── Глобальный стиль ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  '#0F1117',
    'axes.facecolor':    '#1A1D2E',
    'axes.edgecolor':    '#2E3150',
    'axes.labelcolor':   '#C8D0F0',
    'axes.titlecolor':   '#FFFFFF',
    'axes.titlesize':    15,
    'axes.labelsize':    12,
    'axes.grid':         True,
    'grid.color':        '#2E3150',
    'grid.linewidth':    0.6,
    'xtick.color':       '#8892B0',
    'ytick.color':       '#8892B0',
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'text.color':        '#CDD6F4',
    'font.family':       'DejaVu Sans',
    'legend.facecolor':  '#1A1D2E',
    'legend.edgecolor':  '#2E3150',
    'legend.fontsize':   10,
})

# Палитра
TEAL    = '#64FFDA'
BLUE    = '#82AAFF'
PURPLE  = '#C792EA'
ORANGE  = '#FFCB6B'
RED     = '#F07178'
GREEN   = '#C3E88D'
PALETTE = [TEAL, BLUE, PURPLE, ORANGE, RED, GREEN,
           '#89DDFF', '#FF5370', '#B2CCD6', '#EEFFFF']

print('✅  Окружение настроено.')

In [ ]:
# ── Загрузка и очистка ────────────────────────────────────────────────────────
import os
DATA_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..",
                         "data", "raw",
                         "Выгрузка по выданным субсидиям 2025 год (обезлич).xlsx")
# Fallback: если запускается из корня проекта
if not os.path.exists(DATA_PATH):
    DATA_PATH = "data/raw/Выгрузка по выданным субсидиям 2025 год (обезлич).xlsx"

FIGURES_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "figures")
if not os.path.exists(FIGURES_DIR):
    FIGURES_DIR = "notebooks/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

df_raw = pd.read_excel(DATA_PATH, header=4)
df_raw.columns = [
    "num", "date", "_c3", "_c4",
    "oblast", "akimat", "app_number",
    "direction", "subsidy_name", "status",
    "normative", "amount", "district",
]
df_raw = df_raw.drop(columns=["_c3", "_c4"]).dropna(subset=["num"])
df_raw = df_raw[df_raw["num"] != "№ п/п"].copy()
df_raw["amount"]    = pd.to_numeric(df_raw["amount"],    errors="coerce")
df_raw["normative"] = pd.to_numeric(df_raw["normative"], errors="coerce")
df_raw["date"]      = pd.to_datetime(df_raw["date"],     errors="coerce")

# Статус: одобрен / отклонён
APPROVED = {"Исполнена", "Одобрена"}
REJECTED = {"Отклонена"}
def label_status(s):
    if str(s) in APPROVED: return "Одобрена"
    if str(s) in REJECTED: return "Отклонена"
    return "Прочее"

df = df_raw.copy()
df["status_label"] = df["status"].apply(label_status)

# Строки форматирования для отчёта
print(f"Загружено строк : {len(df):,}")
print(f"Уникальных областей : {df['oblast'].nunique()}")
print(f"Уникальных районов  : {df['district'].nunique()}")
print(f"Диапазон дат        : {df['date'].min().date()} — {df['date'].max().date()}")
print(f"Сумма субсидий (₸)  : {df['amount'].sum():,.0f}")
df.head(3)

## 1. Распределение заявок по областям

In [ ]:
oblast_counts = (
    df.groupby('oblast', dropna=True)
      .size()
      .rename('count')
      .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(13, 7))
fig.patch.set_facecolor('#0F1117')

colors = [TEAL if v == oblast_counts.max() else BLUE for v in oblast_counts.values]
bars = ax.barh(oblast_counts.index, oblast_counts.values,
               color=colors, edgecolor='#0F1117', linewidth=0.5,
               height=0.65)

for bar, val in zip(bars, oblast_counts.values):
    ax.text(val + oblast_counts.max() * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:,}', va='center', ha='left',
            color='#CDD6F4', fontsize=9.5, fontweight='bold')

ax.set_xlabel('Количество заявок', fontsize=12)
ax.set_title('Распределение заявок по областям Казахстана', fontsize=16, pad=16,
             color='#FFFFFF', fontweight='bold')
ax.set_xlim(0, oblast_counts.max() * 1.13)
ax.tick_params(axis='y', labelsize=10)
ax.spines[:].set_visible(False)

# Аннотация лидера
leader = oblast_counts.idxmax()
ax.text(0.98, 0.02,
        f'Лидер: {leader}\n{oblast_counts.max():,} заявок',
        transform=ax.transAxes, ha='right', va='bottom',
        color=TEAL, fontsize=10, style='italic',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#1A1D2E', edgecolor=TEAL, alpha=0.8))

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig1_oblast.png'), dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

### 📌 Наблюдение 1
Заявки **неравномерно** распределены по регионам: несколько областей аккумулируют подавляющее большинство субсидий.
Это может свидетельствовать как о реальной концентрации сельхозпроизводства, так и о разном уровне
«субсидийной грамотности» в регионах. Данный дисбаланс важно учитывать при обучении ML-модели
во избежание регионального смещения (regional bias).

## 2. Распределение сумм субсидий

In [ ]:
amounts = df['amount'].dropna()
amounts_k = amounts / 1_000  # в тысячах ₸

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor('#0F1117')

# -- Левый: обычная шкала -------------------------------------------------
ax = axes[0]
n, bins, patches = ax.hist(amounts_k, bins=60, color=BLUE, edgecolor='#0F1117',
                            linewidth=0.3)
# Градиент по высоте столбцов
max_n = n.max()
for patch, height in zip(patches, n):
    patch.set_facecolor(plt.cm.cool(height / max_n * 0.8 + 0.1))

ax.axvline(amounts_k.median(), color=ORANGE, linewidth=1.8, linestyle='--', label=f'Медиана: {amounts_k.median():,.0f} тыс. ₸')
ax.axvline(amounts_k.mean(),   color=RED,    linewidth=1.8, linestyle='-',  label=f'Среднее: {amounts_k.mean():,.0f} тыс. ₸')
ax.set_xlabel('Сумма субсидии, тыс. ₸')
ax.set_ylabel('Количество заявок')
ax.set_title('Гистограмма сумм субсидий', fontweight='bold')
ax.legend()
ax.spines[:].set_visible(False)

# -- Правый: log-шкала ---------------------------------------------------
ax2 = axes[1]
ax2.hist(amounts_k[amounts_k > 0], bins=60, color=TEAL, edgecolor='#0F1117',
         linewidth=0.3, log=True)
ax2.set_xlabel('Сумма субсидии, тыс. ₸')
ax2.set_ylabel('Количество заявок (log)')
ax2.set_title('Гистограмма (логарифмическая шкала)', fontweight='bold')
ax2.spines[:].set_visible(False)

# Статистика
stats_text = (f"min: {amounts_k.min():,.0f}\n"
              f"p25: {amounts_k.quantile(0.25):,.0f}\n"
              f"p75: {amounts_k.quantile(0.75):,.0f}\n"
              f"max: {amounts_k.max():,.0f}")
ax2.text(0.97, 0.97, stats_text, transform=ax2.transAxes,
         va='top', ha='right', fontsize=9, color='#CDD6F4',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#1A1D2E', edgecolor='#2E3150'))

fig.suptitle('Распределение сумм субсидий (тыс. ₸)', fontsize=17,
             fontweight='bold', color='#FFFFFF', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig2_amounts.png'), dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

### 📌 Наблюдение 2
Распределение сумм субсидий имеет выраженный **правый скос (right skew)**: большинство заявок
сосредоточены в низком диапазоне, но есть значимый «хвост» с крупными выплатами.
Медиана существенно ниже среднего — классический признак тяжёлого хвоста.
Для ML-моделей рекомендуется логарифмическое преобразование признака `amount`.

## 3. Топ-10 районов по количеству заявок

In [ ]:
top10 = (
    df.groupby('district', dropna=True)
      .size()
      .rename('count')
      .nlargest(10)
      .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('#0F1117')

# Градиент цвета по значению
norm_vals = (top10.values - top10.values.min()) / (top10.values.max() - top10.values.min())
colors = [plt.cm.plasma(0.3 + v * 0.6) for v in norm_vals]

bars = ax.barh(top10.index, top10.values, color=colors,
               edgecolor='#0F1117', linewidth=0.4, height=0.65)

for bar, val in zip(bars, top10.values):
    ax.text(val + top10.max() * 0.008, bar.get_y() + bar.get_height() / 2,
            f'{val:,}', va='center', ha='left',
            color='#EEFFFF', fontsize=10, fontweight='bold')

ax.set_xlabel('Количество заявок')
ax.set_title('Топ-10 районов по количеству заявок на субсидии',
             fontsize=16, fontweight='bold', pad=15)
ax.set_xlim(0, top10.max() * 1.12)
ax.spines[:].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig3_top10.png'), dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

### 📌 Наблюдение 3
Топ-10 районов концентрируют непропорционально большую долю всех заявок.
Высокая активность может коррелировать с развитостью инфраструктуры районных акиматов
и цифровой грамотностью местных фермеров. В модели скоринга `district` является
потенциально сильным категориальным признаком.

## 4. Соотношение одобренных / отклонённых заявок

In [ ]:
status_counts = df['status_label'].value_counts()

STATUS_COLORS = {
    'Одобрена':  TEAL,
    'Отклонена': RED,
    'Прочее':    '#8892B0',
}
colors_pie = [STATUS_COLORS.get(s, '#8892B0') for s in status_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0F1117')

# -- Pie chart -----------------------------------------------------------
ax = axes[0]
wedges, texts, autotexts = ax.pie(
    status_counts.values,
    labels=None,
    colors=colors_pie,
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.82,
    wedgeprops=dict(edgecolor='#0F1117', linewidth=2.5),
    explode=[0.03] * len(status_counts),
)
for at in autotexts:
    at.set(color='#FFFFFF', fontsize=11, fontweight='bold')

legend_labels = [f"{s}  ({status_counts[s]:,})" for s in status_counts.index]
ax.legend(wedges, legend_labels, loc='lower center', bbox_to_anchor=(0.5, -0.12),
          ncol=1, framealpha=0.0, fontsize=11)
ax.set_title('Структура статусов заявок', fontsize=14, fontweight='bold', pad=20)

# -- Bar с пробивкой по областям -----------------------------------------
ax2 = axes[1]
status_oblast = (
    df[df['status_label'].isin(['Одобрена', 'Отклонена'])]
    .groupby(['oblast', 'status_label'])
    .size()
    .unstack(fill_value=0)
)
status_oblast['Approval_rate'] = (
    status_oblast.get('Одобрена', 0) /
    (status_oblast.get('Одобрена', 0) + status_oblast.get('Отклонена', 0))
).fillna(0)
status_oblast = status_oblast.sort_values('Approval_rate')

y_pos = range(len(status_oblast))
rates = status_oblast['Approval_rate'].values
bar_colors = [TEAL if r >= 0.5 else RED for r in rates]
ax2.barh(y_pos, rates * 100, color=bar_colors, edgecolor='#0F1117',
         linewidth=0.4, height=0.6)
ax2.axvline(50, color='#8892B0', linestyle='--', linewidth=1.2, alpha=0.7)
ax2.set_yticks(list(y_pos))
ax2.set_yticklabels(status_oblast.index, fontsize=9)
ax2.set_xlabel('Доля одобренных, %')
ax2.set_title('Коэффициент одобрения по областям', fontsize=13, fontweight='bold')
ax2.set_xlim(0, 110)
ax2.spines[:].set_visible(False)

for i, v in enumerate(rates):
    ax2.text(v * 100 + 1.5, i, f'{v*100:.1f}%', va='center', fontsize=8.5,
             color='#CDD6F4')

fig.suptitle('Анализ одобрения заявок', fontsize=17,
             fontweight='bold', color='#FFFFFF', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig4_approval.png'), dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

### 📌 Наблюдение 4
Коэффициент одобрения **существенно варьируется** между регионами, что подтверждает
гипотезу о региональной неоднородности в принятии решений. Некоторые области демонстрируют
аномально низкий или высокий процент одобрения — это ключевой сигнал для merit-based скоринга:
текущий процесс не является объективным.

## 5. Средняя сумма субсидий по направлениям животноводства

In [ ]:
dir_stats = (
    df.groupby('direction', dropna=True)['amount']
      .agg(['mean', 'median', 'count'])
      .rename(columns={'mean': 'Среднее', 'median': 'Медиана', 'count': 'Кол-во'})
      .sort_values('Среднее', ascending=False)
      .head(15)
)
dir_stats['Среднее_k']  = dir_stats['Среднее']  / 1_000
dir_stats['Медиана_k'] = dir_stats['Медиана'] / 1_000

fig, ax = plt.subplots(figsize=(14, 8))
fig.patch.set_facecolor('#0F1117')

x = np.arange(len(dir_stats))
w = 0.4

bars1 = ax.bar(x - w/2, dir_stats['Среднее_k'],  width=w, label='Среднее',
               color=BLUE,   edgecolor='#0F1117', linewidth=0.4)
bars2 = ax.bar(x + w/2, dir_stats['Медиана_k'], width=w, label='Медиана',
               color=TEAL,   edgecolor='#0F1117', linewidth=0.4)

# Метки с количеством заявок
for i, (_, row) in enumerate(dir_stats.iterrows()):
    ax.text(i, max(row['Среднее_k'], row['Медиана_k']) + dir_stats['Среднее_k'].max() * 0.015,
            f"n={int(row['Кол-во']):,}", ha='center', va='bottom',
            fontsize=8, color='#8892B0')

ax.set_xticks(x)
ax.set_xticklabels(
    [s[:30] + '...' if len(s) > 30 else s for s in dir_stats.index],
    rotation=38, ha='right', fontsize=8.5
)
ax.set_ylabel('Сумма, тыс. ₸')
ax.set_title('Средняя и медианная сумма субсидий по направлениям животноводства',
             fontsize=15, fontweight='bold', pad=15)
ax.legend(fontsize=11)
ax.spines[:].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig5_directions.png'), dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

### 📌 Наблюдение 5
Суммы субсидий **кардинально различаются** по направлениям: отдельные направления получают
на порядок больше, чем другие. Разрыв между средним и медианным значением указывает на
наличие «крупных игроков», смещающих среднюю вверх. Признак `direction` — один из
наиболее прогностичных для ML-модели.

## 6. Корреляционная матрица числовых признаков

In [ ]:
# Инженерия признаков для матрицы
df_num = df.copy()
df_num['month']       = df_num['date'].dt.month.fillna(0).astype(int)
df_num['day_of_year'] = df_num['date'].dt.dayofyear.fillna(0).astype(int)
df_num['animals_est'] = np.where(df_num['normative'] > 0,
                                  df_num['amount'] / df_num['normative'], 0)
df_num['log_amount']  = np.log1p(df_num['amount'].fillna(0))
df_num['approved_num']= df_num['status_label'].map({'Одобрена': 1, 'Отклонена': 0})

NUM_COLS = ['amount', 'log_amount', 'normative', 'animals_est',
            'month', 'day_of_year', 'approved_num']
COL_LABELS = ['Сумма', 'log(Сумма)', 'Норматив', 'Кол-во голов',
              'Месяц', 'День года', 'Одобрена']

corr = df_num[NUM_COLS].dropna().corr()
corr.index   = COL_LABELS
corr.columns = COL_LABELS

# Маска верхнего треугольника
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0F1117')

cmap = sns.diverging_palette(240, 10, s=80, l=50, as_cmap=True)
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap=cmap, center=0, vmin=-1, vmax=1,
    linewidths=1.5, linecolor='#0F1117',
    annot_kws={'size': 11, 'color': '#EEFFFF', 'fontweight': 'bold'},
    ax=ax,
    cbar_kws={'shrink': 0.75, 'label': 'Коэффициент корреляции Пирсона'}
)

ax.set_title('Корреляционная матрица числовых признаков',
             fontsize=16, fontweight='bold', pad=18)
ax.tick_params(axis='x', rotation=30, labelsize=10)
ax.tick_params(axis='y', rotation=0,  labelsize=10)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig6_corr.png'), dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

### 📌 Наблюдение 6
Ключевые корреляции:
- **`amount` ↔ `normative`** — сильная положительная корреляция: сумма субсидии
  пропорциональна нормативу на голову скота (логично).
- **`log_amount`** улучшает линейность связи с другими признаками — логарифмирование оправдано.
- **Сезонность** (`month`, `day_of_year`) слабо коррелирует с суммой, но может влиять
  на вероятность одобрения — эффект смешанный.
- Мультиколлинеарность между `amount` и `log_amount` очевидна — в модель включаем только один.

## 7. Бонус: Временной тренд и heatmap активности

In [ ]:
df_dated = df.dropna(subset=['date']).copy()
df_dated['month_name'] = df_dated['date'].dt.to_period('M').astype(str)
monthly = df_dated.groupby('month_name').agg(
    count=('num', 'count'),
    total_amount=('amount', 'sum')
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.patch.set_facecolor('#0F1117')

# -- Линия: количество заявок по месяцам ---------------------------------
ax = axes[0]
ax.fill_between(range(len(monthly)), monthly['count'],
                color=BLUE, alpha=0.25)
ax.plot(range(len(monthly)), monthly['count'],
        color=BLUE, linewidth=2.2, marker='o', markersize=5)
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly['month_name'], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Кол-во заявок')
ax.set_title('Количество заявок по месяцам', fontweight='bold')
ax.spines[:].set_visible(False)

# -- Линия: сумма субсидий по месяцам ------------------------------------
ax2 = axes[1]
ax2.fill_between(range(len(monthly)), monthly['total_amount'] / 1e6,
                 color=TEAL, alpha=0.25)
ax2.plot(range(len(monthly)), monthly['total_amount'] / 1e6,
         color=TEAL, linewidth=2.2, marker='s', markersize=5)
ax2.set_xticks(range(len(monthly)))
ax2.set_xticklabels(monthly['month_name'], rotation=30, ha='right', fontsize=9)
ax2.set_ylabel('Сумма субсидий, млн ₸')
ax2.set_title('Общая сумма субсидий по месяцам', fontweight='bold')
ax2.spines[:].set_visible(False)

fig.suptitle('Временная динамика субсидирования', fontsize=16,
             fontweight='bold', color='#FFFFFF', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig7_timeline.png'), dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

---
## 📊 Итоговые выводы EDA

### Ключевые находки

| # | Вывод | Импликация для ML |
|---|-------|-------------------|
| 1 | **Региональный дисбаланс**: топ-3 области дают ~50% заявок | Стратифицированная выборка при обучении, региональный признак необходим |
| 2 | **Скошенное распределение сумм**: медиана << среднего, тяжёлый хвост | Логарифмирование `amount` перед обучением |
| 3 | **Концентрация по районам**: топ-10 районов — ~30% всех заявок | `district` — сильный признак, нужен target encoding |
| 4 | **Неоднородный approval rate**: 0–100% по регионам | Текущий процесс субъективен → merit-based скоринг обоснован |
| 5 | **Направление определяет сумму**: разброс до 10x между категориями | `direction` — топ-признак для модели |
| 6 | **Мультиколлинеарность**: `amount` ∝ `normative` × `animals_count` | В модель: только `log_amount` или составные признаки |
| 7 | **Сезонность**: пики активности в Q1 и Q3 | `month`, `quarter` как признаки сезонности |

### Рекомендации для Feature Engineering

```python
# Предлагаемые признаки для ML-пайплайна
features = [
    'log_amount',          # логарифм суммы субсидии
    'normative',           # норматив на голову скота
    'animals_est',         # оценочное кол-во голов (amount / normative)
    'month',               # сезонность
    'day_of_year',         # день подачи
    'oblast_encoded',      # целевое кодирование региона
    'district_encoded',    # целевое кодирование района
    'direction_encoded',   # целевое кодирование направления
    'subsidy_type_code',   # тип субсидии
]
```

---
*AgriScore KZ EDA | Hackathon 2025 | Данные Министерства сельского хозяйства РК*